Aqui há um projeto para detectar spam de um texto, assim, vemos a importância da variedade dos dados e como um bom modelo de ML pode ser treinado. Assim, a parte de noções e analise do modelo e seu treinamento, regras de negócio, análise da variedade de dados.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, log_loss

In [ ]:

df = pd.read_csv('dataEmail.csv')

textos = df['Texto'] #feature
classificacao = df['Spam'] #label

#transforma a coluna de texto e identifica caracteristicas de acordo com as features
#transforma o texto em uma interpretação matemática!!
vetor = TfidfVectorizer()
matriz = vetor.fit_transform(textos)

X_treino, X_teste, y_treino, y_teste = train_test_split(matriz, classificacao, test_size = 0.2, random_state = 42)

#fazendo o treinamento por épocas
#esse algoritmo emula a regressão lógistica, mas o treinamento ocorre em etapas
model = SGDClassifier(loss = 'log_loss', 
                      learning_rate = 'adaptive', #adapta os passos de treinamento
                      eta0 = 0.1, #tamanho do passo inicial 
                      random_state = 42 
                      ) 

#armazena as perdas
lossHistory = []    
melhorErro = float('inf')
paciencia = 5
epochsNoEvolution = 0


In [3]:
#treino
for epoca in range(1000):
    model.partial_fit(X_treino, y_treino, classes=np.unique(y_treino))
    
    probabilidade = model.predict_proba(X_treino)
    
    erro = log_loss(y_treino, probabilidade)
    
    lossHistory.append(erro)
    
    
    #logica de stop
    if erro < melhorErro - 0.0001:
        melhorErro = erro
        EpochsNoEvolution = 0 
        
    else:
        EpochsNoEvolution += 1
        
    if EpochsNoEvolution >= paciencia:
        print(f"O treinamento parou na época {epoca}")
        break
    
    
    
y_pred = model.predict(X_teste)
acuracia = accuracy_score(y_teste, y_pred)
precision = precision_score(y_teste, y_pred)
recall = recall_score(y_teste, y_pred)
vn, fp, fn, vp = confusion_matrix(y_teste, y_pred).ravel()

#calculo da taxa de falsos positivos
fpr = fp / (fp + vn)

print("-----------------------------")
print(f"acurácia: {acuracia:.2f}")
print(f"precisão: {precision:.2f}")
print(f"recall: {recall:.2f}")
print(f"FPR: {fpr:.2f}")
print("-----------------------------")        

O treinamento parou na época 632
-----------------------------
acurácia: 0.92
precisão: 1.00
recall: 0.83
FPR: 0.00
-----------------------------


In [5]:
textoTeste = "Olá, João! O seu pedido #45892 foi despachado pelos Correios e já está a caminho do seu endereço. Você pode acompanhar a entrega em tempo real acessando a sua conta na nossa plataforma oficial."
with open('arquivo.txt', 'w', encoding='utf-8') as f:
    f.write(textoTeste)

with open('arquivo.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
matrix = vetor.transform([text])
prob = model.predict_proba(matrix)[0][1]

threshold = 0.85

print(f"Chance matemática de ser Spam: {prob * 100:.2f}%")

if prob >= threshold:
    print("Alerta: Spam Detectado")

else: 
    print("Email Seguro")    

Chance matemática de ser Spam: 28.20%
Email Seguro
